# Notebook 02 - Tokenization and Vectorization (run on GOOGLE COLAB)
**Goal:** train BPE + WordPiece tokenizers, TF-IDF, Word2Vec and create BERT vectors. Save everything for the API.

**Before you start (Colab):** Runtime > Change runtime type > **T4 GPU** (makes BERT fast).

**Files you will upload when asked:** `bbc_clean.csv`, `train.csv`, `test.csv` (from `data/processed/`) and `tokenization.py` (from `app/services/`).

**Output:** `artifacts.zip` - unzip its content into the project's `models/` folder.

In [ ]:
# Install the libraries Colab does not have (restart runtime if Colab asks)
!pip install -q gensim

In [ ]:
# Import the libraries we need
import json  # json saves the word list
import os  # os helps with folders
import shutil  # shutil can create zip files

import joblib  # saves .pkl files
import numpy as np  # numpy handles number arrays
import pandas as pd  # pandas works with tables
import torch  # PyTorch is needed by BERT
from google.colab import files  # lets us upload and download files in Colab
from gensim.models import Word2Vec  # Word2Vec model
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF model
from transformers import AutoModel, AutoTokenizer  # BERT model and tokenizer

## Step 1 - Upload the data files

In [ ]:
# Select 4 files: bbc_clean.csv, train.csv, test.csv, tokenization.py
uploaded = files.upload()  # opens a file picker
print(list(uploaded.keys()))  # show what was uploaded

In [ ]:
# Load the data and import our tokenizer code
import tokenization  # the tokenization.py file we just uploaded
from tokenization import BPETokenizer, WordPieceTokenizer  # our two tokenizer classes

df = pd.read_csv("bbc_clean.csv")  # all documents
train_df = pd.read_csv("train.csv")  # training documents
test_df = pd.read_csv("test.csv")  # test documents
os.makedirs("artifacts", exist_ok=True)  # folder where we collect all saved files
print(df.shape, train_df.shape, test_df.shape)  # check sizes

## Step 2 (Day 1) - Train and compare BPE and WordPiece (this takes a few minutes)

In [ ]:
# Train both tokenizers on the training texts
train_texts = list(train_df["clean_text"])  # texts used for learning
bpe = BPETokenizer(num_merges=300)  # create a BPE tokenizer
bpe.train(train_texts)  # learn the merges
wordpiece = WordPieceTokenizer(num_merges=300)  # create a WordPiece tokenizer
wordpiece.train(train_texts)  # learn the vocabulary
print("Training finished")  # confirm

In [ ]:
# Compare vocabulary size, token count and unknown words on the TEST texts
rows = []  # one result row per tokenizer
for name, tok in [("BPE", bpe), ("WordPiece", wordpiece)]:  # loop over both tokenizers
    tokens = []  # all tokens of all test texts
    for text in test_df["clean_text"]:  # tokenize every test text
        tokens += tok.tokenize(text)  # collect the tokens
    unknown = sum(1 for t in tokens if t in ("<unk>", "[UNK]"))  # count unknown tokens
    rows.append({"tokenizer": name, "vocab_size": len(tok.vocab), "token_count": len(tokens), "unknown_tokens": unknown})  # save numbers
tokenizer_comparison = pd.DataFrame(rows)  # make a table
tokenizer_comparison  # show the table

In [ ]:
# See how each tokenizer handles words it never saw
sample = "football blockchain zzzqx"  # one known word, one rare word, one nonsense word
print("BPE      :", bpe.tokenize(sample))  # BPE result
print("WordPiece:", wordpiece.tokenize(sample))  # WordPiece result

In [ ]:
# Save the tokenizers and the comparison table
bpe.save("artifacts/bpe_tokenizer.json")  # save BPE
wordpiece.save("artifacts/wordpiece_tokenizer.json")  # save WordPiece
tokenizer_comparison.to_csv("artifacts/tokenizer_comparison.csv", index=False)  # save comparison for the report

## Step 3 (Day 2) - TF-IDF

In [ ]:
# Train TF-IDF on all cleaned documents (5000 most useful words)
tfidf = TfidfVectorizer(max_features=5000)  # create the vectorizer
tfidf.fit(df["clean_text"])  # learn vocabulary and word weights
joblib.dump(tfidf, "artifacts/tfidf_vectorizer.pkl")  # save as pickle file
print("TF-IDF vocabulary size:", len(tfidf.vocabulary_))  # confirm

## Step 4 (Day 2) - Word2Vec

In [ ]:
# Train Word2Vec on all documents
sentences = [text.split() for text in df["clean_text"]]  # each document = list of words
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=2, epochs=20, seed=42)  # train
np.save("artifacts/word2vec_vectors.npy", w2v.wv.vectors)  # save all word vectors as numbers
with open("artifacts/word2vec_vocab.json", "w") as file:  # save the word list (same order as the vectors)
    json.dump(list(w2v.wv.index_to_key), file)  # write words as JSON
print("Word2Vec words:", len(w2v.wv))  # confirm
print(w2v.wv.most_similar("football", topn=5))  # words close to "football"

## Step 5 (Day 2) - BERT embeddings for every document

In [ ]:
# Load pretrained BERT (use GPU if Colab gives us one)
device = "cuda" if torch.cuda.is_available() else "cpu"  # choose GPU or CPU
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # BERT tokenizer
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()  # BERT model
print("Running on:", device)  # show the device

In [ ]:
# Function: turn a list of texts into BERT vectors (average of all token vectors)
def encode_batch(texts):
    inputs = bert_tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)  # text -> ids
    with torch.no_grad():  # no training
        hidden = bert_model(**inputs).last_hidden_state  # vector for every token
    mask = inputs["attention_mask"].unsqueeze(-1)  # 1 for real tokens, 0 for padding
    summed = (hidden * mask).sum(dim=1)  # add up real token vectors
    return (summed / mask.sum(dim=1)).cpu().numpy()  # average = one vector per text

In [ ]:
# Encode all documents in small batches (use the RAW text, BERT likes natural text)
texts = list(df["text"])  # original articles
batches = [encode_batch(texts[i:i + 32]) for i in range(0, len(texts), 32)]  # 32 texts at a time
bert_vectors = np.vstack(batches).astype("float32")  # join all batches
np.save("artifacts/bert_doc_vectors.npy", bert_vectors)  # save the vectors
print("BERT vectors shape:", bert_vectors.shape)  # (number of documents, 768)

## Step 6 - Quick check: top 5 similar documents with each method

In [ ]:
# Cosine similarity written with numpy
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))  # (a . b) / (|a| |b|)

query = "The football team won the championship match"  # example query
clean_query = " ".join(w for w in query.lower().split())  # simple clean version
query_vectors = {  # query vector for each method
    "tfidf": tfidf.transform([clean_query]).toarray()[0],
    "word2vec": np.mean([w2v.wv[w] for w in clean_query.split() if w in w2v.wv], axis=0),
    "bert": encode_batch([query])[0],
}
doc_vectors = {  # document vectors for each method
    "tfidf": tfidf.transform(df["clean_text"]).toarray(),
    "word2vec": np.array([np.mean([w2v.wv[w] for w in t.split() if w in w2v.wv], axis=0) for t in df["clean_text"]]),
    "bert": bert_vectors,
}
for method in ["tfidf", "word2vec", "bert"]:  # test every method
    scores = [cosine(query_vectors[method], row) for row in doc_vectors[method]]  # compare with all documents
    top5 = np.argsort(scores)[::-1][:5]  # best 5 positions
    print(method, "->", list(df.loc[top5, "label"]))  # labels of the best 5

## Step 7 - Download everything

In [ ]:
# Zip the artifacts folder and download it
shutil.make_archive("artifacts", "zip", "artifacts")  # create artifacts.zip
files.download("artifacts.zip")  # download to your computer

**Done!** Unzip `artifacts.zip` into your project's `models/` folder. Next: notebook 03.